In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_Res
from train import WarmUpCosine, CustomWeightDecaySGD, RSquared
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_reg import load_wb_if_exists
from evaluate_reg import evalu_stream_main_selected, evalu_select, eval_acc_select_list_single_thresholds, evalu_prepare, compute_stats

In [ ]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [5]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()

✔ 检测到缓存数据，已直接加载


In [6]:
model=load_Res()

In [7]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("re_lu_16").output,
    outputs=model.output
)

In [8]:
l_label=[3, 11, 18, 27, 34, 43, 50, 59, 66]

In [9]:
layer_list = [model.layers[l].name for l in l_label]

In [10]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data/Res-18/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data/Res-18/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data/Res-18/layer_cache/salt/"+str(i))
    save_layer_outputs_and_labels(model, x_move, y_train, layer_list, save_dir="D:/Data/Res-18/layer_cache/move/"+str(i))
    save_layer_outputs_and_labels(model, x_occ, y_train, layer_list, save_dir="D:/Data/Res-18/layer_cache/occ/"+str(i))

[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/base
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/gauss/0
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/salt/0
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/move/0
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/occ/0
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/gauss/1
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/salt/1
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/move/1
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/occ/1
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/gauss/2
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/salt/2
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/move/2
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/occ/2
[SKIP] All layer files already exist in 

In [11]:
for i in range(5):
    path = "./attack/" + str(i)
    ATTACK_DIR = os.path.join(path, "Res_pgd.npy")
    x_attack = np.load(ATTACK_DIR)
    save_layer_outputs_and_labels(model, x_attack, y_train, layer_list, save_dir="D:/Data/Res-18/layer_cache/attack/"+str(i))

[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/attack/0
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/attack/1
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/attack/2
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/attack/3
[SKIP] All layer files already exist in D:/Data/Res-18/layer_cache/attack/4


In [12]:
CACHE_DIR = "./RES-18/w_and_b_cache"

In [13]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data/Res-18/layer_cache/base")

In [14]:
threshold_list, Y_list = evalu_prepare(y_train, n=9)

In [15]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data/Res-18/layer_cache/base")

In [16]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data/Res-18/layer_cache/base")
base_final = eval_acc_select_list_single_thresholds(model, x_train, 'train', select_list, threshold_list) 
base = np.concatenate((base,base_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.89873105 0.7296615  0.65220665 0.62967278 0.62082748 0.54029827
 0.56433068 0.65092579 0.86320829]
Layer 1
accuracy: [0.94965235 0.82175878 0.68967522 0.63788336 0.70302096 0.63430965
 0.6156881  0.71795014 0.78183348]
Layer 2
accuracy: [0.95551247 0.84403822 0.72879358 0.66904107 0.73069085 0.67336915
 0.63989584 0.73391575 0.7981126 ]
Layer 3
accuracy: [0.97112913 0.88995115 0.77025807 0.71991735 0.78046414 0.73010245
 0.69250041 0.74179001 0.88728063]
Layer 4
accuracy: [0.9687467  0.89512628 0.7764433  0.76533343 0.82226614 0.76584336
 0.69685948 0.75249899 0.83943217]
Layer 5
accuracy: [0.95745729 0.95552855 0.87693459 0.8561714  0.85618136 0.87070391
 0.85611791 0.89256959 0.93882969]
Layer 6
accuracy: [0.94673545 0.96751965 0.93115778 0.87415625 0.86739579 0.90553241
 0.89466872 0.9135152  0.95110216]
Layer 7
accuracy: [0.94998788 0.97511858 0.97490659 0.94355171 0.93255611 0.94058093
 0.93013224 0.933442   0.96252409]
Layer 8
accuracy: [0.94976906 0.96852217

In [17]:
compute_stats(base)

(array([[0.76019973, 0.59693284, 0.69282159],
        [0.82036212, 0.65840465, 0.70515724],
        [0.84278142, 0.69103369, 0.72397473],
        [0.87711278, 0.74349464, 0.77385702],
        [0.88010543, 0.78448097, 0.76293022],
        [0.92997348, 0.86101889, 0.89583906],
        [0.94847096, 0.88236148, 0.91976203],
        [0.96667102, 0.93889625, 0.94203278],
        [0.9376425 , 0.86487225, 0.86658838],
        [1.        , 1.        , 1.        ]]),
 array([[0.10293366, 0.04020917, 0.12556114],
        [0.10613981, 0.03158221, 0.06842912],
        [0.09256186, 0.02809746, 0.06497309],
        [0.08250621, 0.02647   , 0.08268838],
        [0.07922276, 0.02671896, 0.05867055],
        [0.03751242, 0.00684835, 0.03384599],
        [0.01489531, 0.01661515, 0.02345848],
        [0.01179708, 0.00464431, 0.01455241],
        [0.03135891, 0.02686582, 0.14320574],
        [0.        , 0.        , 0.        ]]))

In [18]:
attack = np.zeros((len(layer_list),9))
attack_final = np.zeros(9)
for i in range(5):
    attack += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/Res-18/layer_cache/attack/"+str(i))
attack = attack/5
for i in range(5):
    path = "./attack/" + str(i)
    ATTACK_DIR = os.path.join(path, "Res_pgd.npy")
    attack_final += eval_acc_select_list_single_thresholds(model, ATTACK_DIR, 'train', select_list, threshold_list)
attack_final = attack_final/5
attack = np.concatenate((attack,attack_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.87284607 0.70439447 0.63239098 0.60794227 0.61821986 0.51149811
 0.60474992 0.69808203 0.89024118]
Layer 1
accuracy: [0.97776129 0.82018732 0.69926503 0.63805321 0.61086336 0.51993308
 0.50957021 0.63765383 0.68269071]
Layer 2
accuracy: [0.97331171 0.7979917  0.69467752 0.61492983 0.55352032 0.43146529
 0.42121776 0.60406395 0.61113268]
Layer 3
accuracy: [0.93464421 0.74157005 0.66205713 0.59983351 0.53060012 0.36585088
 0.26958861 0.17964274 0.10114101]
Layer 4
accuracy: [0.91313652 0.729416   0.65526852 0.59687653 0.52904417 0.3611014
 0.25602894 0.17805997 0.08527766]
Layer 5
accuracy: [0.52402701 0.41703234 0.49368456 0.59687653 0.52904417 0.36065529
 0.24655083 0.15897498 0.06981603]
Layer 6
accuracy: [0.0529542  0.1775282  0.25845328 0.3381469  0.52904417 0.36065529
 0.24655083 0.17558583 0.08172636]
Layer 7
accuracy: [0.07199407 0.17690579 0.25845328 0.33402642 0.52904417 0.36065529
 0.24655083 0.15652219 0.08173658]
Layer 8
accuracy: [0.02718596 0.17649205 

In [19]:
compute_stats(attack)

(array([[0.73654384, 0.57922008, 0.73102438],
        [0.83240455, 0.58961655, 0.60997158],
        [0.82199364, 0.53330515, 0.54547146],
        [0.7794238 , 0.49876151, 0.18345746],
        [0.76594035, 0.49567403, 0.17312219],
        [0.47824797, 0.49552533, 0.15844728],
        [0.16297856, 0.40928212, 0.16795434],
        [0.16911772, 0.40790863, 0.1616032 ],
        [0.15397015, 0.37387312, 0.16235898],
        [0.76141076, 0.49706691, 0.15503688]]),
 array([[0.10076325, 0.04807013, 0.11885625],
        [0.11402335, 0.05050849, 0.0733367 ],
        [0.11501108, 0.0762509 , 0.08790802],
        [0.114457  , 0.09814017, 0.06882133],
        [0.10839587, 0.09910483, 0.06979631],
        [0.04502356, 0.09930677, 0.07215265],
        [0.08452312, 0.08518164, 0.06750534],
        [0.07632059, 0.08634288, 0.06738099],
        [0.09565927, 0.03875087, 0.06634179],
        [0.10388605, 0.10054873, 0.07564001]]))

In [20]:
gauss = np.zeros((len(layer_list),9))
gauss_final = np.zeros(9)
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/Res-18/layer_cache/gauss/"+str(i))
gauss = gauss/10
for i in range(10):
    path = "./noise/" + str(i)
    GAUSS_DIR = os.path.join(path, "gauss.npy")
    gauss_final += eval_acc_select_list_single_thresholds(model, GAUSS_DIR, 'train', select_list, threshold_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.44218361 0.38703684 0.37638873 0.42653036 0.53523331 0.39313404
 0.70887656 0.81242651 0.91559289]
Layer 1
accuracy: [0.98527857 0.82904084 0.73578217 0.65974508 0.5953155  0.53831957
 0.5986796  0.73040777 0.76035953]
Layer 2
accuracy: [0.98440639 0.82679143 0.74115878 0.6658569  0.57897109 0.49340591
 0.56013461 0.72311368 0.73023347]
Layer 3
accuracy: [0.98417818 0.82885276 0.74151225 0.6642957  0.57447326 0.39533125
 0.29795087 0.22117151 0.18018296]
Layer 4
accuracy: [0.9835204  0.82515145 0.74157859 0.66500677 0.5777559  0.42017753
 0.41124768 0.50538495 0.51949373]
Layer 5
accuracy: [0.98047505 0.83160825 0.68426829 0.67978194 0.57979974 0.39313404
 0.29112344 0.18757349 0.08440711]
Layer 6
accuracy: [0.98219253 0.82421188 0.49191533 0.61253612 0.59391457 0.39335411
 0.29112344 0.18757349 0.08440711]
Layer 7
accuracy: [0.983306   0.83297262 0.71228877 0.66769171 0.60330045 0.40339871
 0.29217137 0.18757349 0.08935818]
Layer 8
accuracy: [0.983306   0.82721086

In [21]:
compute_stats(gauss)

(array([[0.40192303, 0.45119711, 0.81229866],
        [0.84948508, 0.59840028, 0.69947004],
        [0.85093986, 0.57939105, 0.67277721],
        [0.85170459, 0.54480823, 0.23506994],
        [0.85012522, 0.55415503, 0.48153401],
        [0.83327351, 0.54890251, 0.1877221 ],
        [0.7666979 , 0.53425539, 0.18788609],
        [0.84658615, 0.55730407, 0.19052161],
        [0.84098679, 0.56278034, 0.2606432 ],
        [0.8500975 , 0.5498227 , 0.44204975]]),
 array([[0.02917849, 0.06064805, 0.08439164],
        [0.10344527, 0.0492273 , 0.07130303],
        [0.100582  , 0.06979162, 0.07777085],
        [0.09992526, 0.1126935 , 0.04610194],
        [0.10001829, 0.1012731 , 0.04726777],
        [0.11804146, 0.11715522, 0.08436622],
        [0.20381347, 0.10037333, 0.08416561],
        [0.10637745, 0.11314626, 0.08209173],
        [0.11062607, 0.10293762, 0.05081696],
        [0.09969745, 0.11150218, 0.21422621]]))

In [22]:
salt = np.zeros((len(layer_list),9))
salt_final = np.zeros(9)
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/Res-18/layer_cache/salt/"+str(i))
salt = salt/10
for i in range(10):
    path = "./noise/" + str(i)
    SALT_DIR = os.path.join(path, "salt.npy")
    salt_final += eval_acc_select_list_single_thresholds(model, SALT_DIR, 'train', select_list, threshold_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.43145945 0.38188928 0.38285213 0.43053537 0.54236266 0.39581995
 0.70802727 0.81221174 0.91559289]
Layer 1
accuracy: [0.97869385 0.81601922 0.69816959 0.65622162 0.62752421 0.48730578
 0.50730913 0.63595604 0.63261265]
Layer 2
accuracy: [0.85281844 0.5990021  0.50979064 0.51357751 0.65400376 0.39755274
 0.30037124 0.23207191 0.14680429]
Layer 3
accuracy: [0.8030614  0.4699302  0.40305373 0.4967458  0.63617763 0.39379945
 0.29135175 0.18778826 0.08564164]
Layer 4
accuracy: [0.92511974 0.62899666 0.50350807 0.55654612 0.63675634 0.40406124
 0.2966659  0.20245586 0.10715142]
Layer 5
accuracy: [0.97895907 0.79981756 0.58358914 0.6179445  0.61501114 0.40210989
 0.29380316 0.1918755  0.1570881 ]
Layer 6
accuracy: [0.98330894 0.8165222  0.41796307 0.47482921 0.60246611 0.41895069
 0.30247161 0.20237432 0.21203457]
Layer 7
accuracy: [0.98352114 0.82433805 0.69842278 0.63379822 0.59398001 0.47765382
 0.3686298  0.24805769 0.28524028]
Layer 8
accuracy: [0.98352114 0.82514165

In [23]:
compute_stats(salt)

(array([[0.39694031, 0.45604813, 0.8120979 ],
        [0.83289487, 0.58711045, 0.59412205],
        [0.65479775, 0.51979382, 0.22479831],
        [0.56473472, 0.50707788, 0.18804679],
        [0.69118034, 0.52793926, 0.20079336],
        [0.78829639, 0.54450678, 0.21421861],
        [0.74071139, 0.4999733 , 0.24204089],
        [0.83455738, 0.57010429, 0.29791597],
        [0.83461516, 0.58138514, 0.4651375 ],
        [0.84611158, 0.58150553, 0.64653803]]),
 array([[0.02485503, 0.06282364, 0.08458497],
        [0.11297187, 0.0738428 , 0.05881555],
        [0.14424584, 0.1021655 , 0.06396374],
        [0.17296847, 0.09595122, 0.08410149],
        [0.17671558, 0.09482128, 0.07797514],
        [0.16427845, 0.10014262, 0.05824026],
        [0.23548285, 0.0740123 , 0.04427984],
        [0.11798739, 0.06097382, 0.05055392],
        [0.11769311, 0.05369751, 0.14327778],
        [0.10483352, 0.06912727, 0.16409712]]))

In [24]:
move = np.zeros((len(layer_list),9))
move_final = np.zeros(9)
for i in range(10):
    move += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/Res-18/layer_cache/move/"+str(i))
move = move/10
for i in range(10):
    path = "./noise/" + str(i)
    MOVE_DIR = os.path.join(path, "move.npy")
    move_final += eval_acc_select_list_single_thresholds(model, MOVE_DIR, 'train', select_list, threshold_list)
move_final = move_final/10
move = np.concatenate((move,move_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.97268522 0.8237063  0.73504147 0.66535444 0.60806642 0.6003482
 0.34895867 0.32948181 0.66471374]
Layer 1
accuracy: [0.97913573 0.82448452 0.73331871 0.66233376 0.63023855 0.61027557
 0.70264345 0.8097217  0.90242595]
Layer 2
accuracy: [0.9828387  0.82638133 0.74136701 0.66597452 0.63334501 0.61244578
 0.70760032 0.81288066 0.91350902]
Layer 3
accuracy: [0.983306   0.82534512 0.7424013  0.66396802 0.65408106 0.64144779
 0.7153783  0.81166383 0.91539241]
Layer 4
accuracy: [0.98309232 0.82554034 0.73271966 0.653445   0.6367625  0.65996326
 0.70472863 0.80005961 0.89686288]
Layer 5
accuracy: [0.9833176  0.82801446 0.7060323  0.6392726  0.61555912 0.54868186
 0.48609552 0.47917716 0.7395667 ]
Layer 6
accuracy: [0.98331833 0.83545027 0.68956258 0.61536436 0.62227405 0.54493568
 0.49210196 0.46678234 0.71948204]
Layer 7
accuracy: [0.98309159 0.83174997 0.73174565 0.65209999 0.61160997 0.5713513
 0.54552719 0.53123935 0.76906626]
Layer 8
accuracy: [0.98309159 0.82988116 0

In [25]:
compute_stats(move)

(array([[0.84307145, 0.62408541, 0.44963004],
        [0.84484771, 0.63512515, 0.80507571],
        [0.85045308, 0.63483343, 0.81181957],
        [0.85027479, 0.65617499, 0.81332983],
        [0.84833222, 0.65232966, 0.79862911],
        [0.84116753, 0.60084359, 0.56841251],
        [0.83901244, 0.59392604, 0.55948792],
        [0.85008182, 0.6128397 , 0.6149072 ],
        [0.84892518, 0.60154084, 0.6343978 ],
        [0.85119241, 0.60889225, 0.67232211]]),
 array([[0.09803429, 0.02855142, 0.15424456],
        [0.10220772, 0.02007188, 0.08118525],
        [0.0999761 , 0.02233804, 0.0839323 ],
        [0.09986224, 0.00956945, 0.08254539],
        [0.10232194, 0.0099464 , 0.08099649],
        [0.11159397, 0.04445552, 0.12023911],
        [0.11634645, 0.03730054, 0.11476951],
        [0.10184282, 0.03789887, 0.10901935],
        [0.10249484, 0.0579304 , 0.1538647 ],
        [0.10414578, 0.03694156, 0.1323842 ]]))

In [26]:
occ = np.zeros((len(layer_list),9))
occ_final = np.zeros(9)
for i in range(10):
    occ += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/Res-18/layer_cache/occ/"+str(i))
occ = occ/10
for i in range(10):
    path = "./noise/" + str(i)
    OCC_DIR = os.path.join(path, "occ.npy")
    occ_final += eval_acc_select_list_single_thresholds(model, OCC_DIR, 'train', select_list, threshold_list)
occ_final = occ_final/10
occ = np.concatenate((occ,occ_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.61619241 0.4450117  0.38588962 0.39174648 0.44394657 0.44939017
 0.70297103 0.81221174 0.91559289]
Layer 1
accuracy: [0.895834   0.71837781 0.55921381 0.52217619 0.55674303 0.58584131
 0.56613397 0.64307354 0.71583265]
Layer 2
accuracy: [0.90758135 0.71545682 0.59176476 0.50557829 0.57385526 0.59466346
 0.52989004 0.60288855 0.69562804]
Layer 3
accuracy: [0.96741163 0.80942578 0.65856024 0.58773785 0.61461371 0.65207182
 0.64919701 0.70236632 0.86766321]
Layer 4
accuracy: [0.92653044 0.69060605 0.53233115 0.52707316 0.57818451 0.60656578
 0.5036585  0.44811447 0.60610869]
Layer 5
accuracy: [0.98025199 0.84138223 0.69009188 0.60168118 0.58643215 0.6284235
 0.68140591 0.78355075 0.91706043]
Layer 6
accuracy: [0.97914146 0.84594987 0.66551722 0.53608145 0.55188785 0.6347335
 0.70695204 0.80251588 0.91502321]
Layer 7
accuracy: [0.98043952 0.84491424 0.77165552 0.67338683 0.59789399 0.62525131
 0.70928357 0.79643806 0.91211098]
Layer 8
accuracy: [0.98022731 0.84243812 0

In [27]:
compute_stats(occ)

(array([[0.48223407, 0.43413194, 0.8099962 ],
        [0.724503  , 0.5599718 , 0.63650153],
        [0.73721401, 0.56024168, 0.60321344],
        [0.80890258, 0.62060055, 0.73988473],
        [0.71351   , 0.57325302, 0.51631243],
        [0.83672866, 0.61116271, 0.79411094],
        [0.82739226, 0.57836165, 0.80630422],
        [0.86583694, 0.6369866 , 0.80456293],
        [0.85625926, 0.62473988, 0.80857683],
        [0.86958862, 0.63299593, 0.82128062]]),
 array([[0.09639638, 0.02968089, 0.08703125],
        [0.13737319, 0.03080269, 0.06043921],
        [0.13212178, 0.0428258 , 0.06637286],
        [0.12874429, 0.03367656, 0.09157697],
        [0.16274446, 0.03910599, 0.06117348],
        [0.11989504, 0.02344513, 0.09388711],
        [0.13250334, 0.0478164 , 0.08569865],
        [0.0866064 , 0.02982371, 0.08427117],
        [0.09604409, 0.03737395, 0.07786755],
        [0.09245649, 0.02289128, 0.0796052 ]]))

In [28]:
SAVE_FILE='Res-18.pkl'

In [29]:
progress = {"base": base, "attack": attack,"gauss": gauss,"salt": salt,"move": move,"occ": occ}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [ ]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=6): 
    """ 
    Input: 
        matrix_dict: {name: np.ndarray(m, n)}, a dictionary containing 6 matrices 
    Output: 
        stats: { 
            name: { 
                "mean": (m,), 
                "min":  (m,), 
                "max":  (m,) 
            }, ... 
        } 
    For each component i, statistics are computed along axis=1 (over n samples): 
        mean[i] = mean(X[i, :]) 
        min[i]  = min(X[i, :]) 
        max[i]  = max(X[i, :]) 
    """ 
    if check_num is not None and len(matrix_dict) != check_num: 
        raise ValueError(f"Expected {check_num} matrices, but received {len(matrix_dict)}") 
 
    # shape consistency 
    shapes = [np.asarray(v).shape for v in matrix_dict.values()] 
    if len(set(shapes)) != 1: 
        raise ValueError(f"Inconsistent matrix shapes: {shapes}") 
 
    stats = {} 
    for name, X in matrix_dict.items(): 
        X = np.asarray(X) 
        stats[name] = { 
            "mean": X.mean(axis=1), 
            "std":  X.std(axis=1), 
            "min":  X.min(axis=1), 
            "max":  X.max(axis=1), 
        } 
    return stats 

In [32]:
SAVE_FILE='Res-18.pkl'
with open(SAVE_FILE, "rb") as f:
    progress = pickle.load(f)

In [33]:
mean_var = stats_minmax_from_matrix_dict(progress)

In [34]:
mean_var

{'base': {'mean': array([0.68331805, 0.72797467, 0.75259662, 0.79815481, 0.80917221,
         0.89561048, 0.91686482, 0.94920002, 0.88970104, 1.        ]),
  'std': array([0.11753188, 0.10139313, 0.09366816, 0.08973625, 0.07791423,
         0.04073115, 0.03289405, 0.01668618, 0.09248799, 0.        ]),
  'min': array([0.54029827, 0.6156881 , 0.63989584, 0.69250041, 0.69685948,
         0.85611791, 0.86739579, 0.93013224, 0.66411708, 1.        ]),
  'max': array([0.89873105, 0.94965235, 0.95551247, 0.97112913, 0.9687467 ,
         0.95745729, 0.96751965, 0.97511858, 0.97180591, 1.        ])},
 'attack': {'mean': array([0.68226277, 0.67733089, 0.63359009, 0.48721425, 0.47824552,
         0.37740686, 0.24673834, 0.24620985, 0.23006741, 0.47117152]),
  'std': array([0.11906959, 0.13809372, 0.16338852, 0.26156976, 0.25988156,
         0.17239408, 0.13976187, 0.1379228 , 0.12397428, 0.26550221]),
  'min': array([0.51149811, 0.50957021, 0.42121776, 0.10114101, 0.08527766,
         0.06981603, 